In [1]:
import pandas as pd
import numpy as np

# Column names for CMAPSS dataset
columns = ['engine_id', 'cycle', 'setting1', 'setting2', 'setting3'] + \
          [f'sensor{i}' for i in range(1, 22)]

# Load training data
train_df = pd.read_csv('../data/raw/train_FD001.txt', sep=' ', header=None)
train_df.drop(columns=[26, 27], inplace=True)
train_df.columns = columns

# Load test data
test_df = pd.read_csv('../data/raw/test_FD001.txt', sep=' ', header=None)
test_df.drop(columns=[26, 27], inplace=True)
test_df.columns = columns

# Load true RUL values
rul_df = pd.read_csv('../data/raw/RUL_FD001.txt', sep=' ', header=None)
rul_df.drop(columns=[1], inplace=True)
rul_df.columns = ['RUL']

print("✅ Data Loaded Successfully!")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", rul_df.shape)
print("\nFirst 5 rows of training data:")
train_df.head()

✅ Data Loaded Successfully!
Train shape: (20631, 26)
Test shape: (13096, 26)
RUL shape: (100, 1)

First 5 rows of training data:


,engine_id,cycle,setting1,setting2,setting3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [2]:
# Add RUL (Remaining Useful Life) labels to training data
# RUL = max cycle for that engine - current cycle

rul_labels = train_df.groupby('engine_id')['cycle'].max().reset_index()
rul_labels.columns = ['engine_id', 'max_cycle']

train_df = train_df.merge(rul_labels, on='engine_id', how='left')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']
train_df.drop(columns=['max_cycle'], inplace=True)

print("✅ RUL Labels Added!")
print("Max RUL:", train_df['RUL'].max())
print("Min RUL:", train_df['RUL'].min())
print("\nSample data with RUL column:")
train_df[['engine_id', 'cycle', 'RUL']].head(10)

✅ RUL Labels Added!
Max RUL: 361
Min RUL: 0

Sample data with RUL column:


,engine_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187
5,1,6,186
6,1,7,185
7,1,8,184
8,1,9,183
9,1,10,182


In [3]:
from sklearn.preprocessing import MinMaxScaler

# Define sensor columns
sensor_cols = [f'sensor{i}' for i in range(1, 22)]
setting_cols = ['setting1', 'setting2', 'setting3']

# Drop sensors that are constant (useless for prediction)
# These sensors have zero variance in FD001
drop_sensors = ['sensor1', 'sensor5', 'sensor6', 'sensor10', 
                'sensor16', 'sensor18', 'sensor19']

useful_sensors = [s for s in sensor_cols if s not in drop_sensors]

print(f"✅ Total sensors: {len(sensor_cols)}")
print(f"❌ Dropped (constant) sensors: {len(drop_sensors)}")
print(f"✅ Useful sensors remaining: {len(useful_sensors)}")

# Normalize sensor values between 0 and 1
scaler = MinMaxScaler()
train_df[useful_sensors] = scaler.fit_transform(train_df[useful_sensors])
test_df[useful_sensors] = scaler.transform(test_df[useful_sensors])

# Cap RUL at 125 (industry standard - engines rarely need prediction beyond this)
train_df['RUL'] = train_df['RUL'].clip(upper=125)

print("\n✅ Normalization Done!")
print("✅ RUL capped at 125 cycles")
print("\nSample normalized data:")
train_df[['engine_id', 'cycle', 'RUL'] + useful_sensors[:3]].head()

✅ Total sensors: 21
❌ Dropped (constant) sensors: 7
✅ Useful sensors remaining: 14

✅ Normalization Done!
✅ RUL capped at 125 cycles

Sample normalized data:


,engine_id,cycle,RUL,sensor2,sensor3,sensor4
0,1,1,125,0.183735,0.406802,0.309757
1,1,2,125,0.283133,0.453019,0.352633
2,1,3,125,0.343373,0.369523,0.370527
3,1,4,125,0.343373,0.256159,0.331195
4,1,5,125,0.349398,0.257467,0.404625


In [4]:
import os

# Save processed data to data/processed/ folder
train_df.to_csv('../data/processed/train_processed.csv', index=False)
test_df.to_csv('../data/processed/test_processed.csv', index=False)
rul_df.to_csv('../data/processed/rul_fd001.csv', index=False)

# Save useful sensor list for later use
import json
with open('../data/processed/useful_sensors.json', 'w') as f:
    json.dump(useful_sensors, f)

print("✅ Processed data saved!")
print("📁 Files saved in data/processed/:")
print("   - train_processed.csv")
print("   - test_processed.csv")
print("   - rul_fd001.csv")
print("   - useful_sensors.json")

✅ Processed data saved!
📁 Files saved in data/processed/:
   - train_processed.csv
   - test_processed.csv
   - rul_fd001.csv
   - useful_sensors.json


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import numpy as np

# Prepare features and target
feature_cols = useful_sensors + ['cycle']
X_train = train_df[feature_cols]
y_train = train_df['RUL']

# Prepare test data
X_test = test_df.groupby('engine_id').last().reset_index()[feature_cols]
y_test = rul_df['RUL'].values

print("✅ Data ready for training!")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test engines: {X_test.shape[0]}")
print("\n⏳ Training models... please wait...")

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test).clip(0, 125)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_mae = mean_absolute_error(y_test, lr_preds)

print(f"\n✅ Linear Regression → RMSE: {lr_rmse:.2f} | MAE: {lr_mae:.2f}")

# 2. Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test).clip(0, 125)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae = mean_absolute_error(y_test, rf_preds)

print(f"✅ Random Forest     → RMSE: {rf_rmse:.2f} | MAE: {rf_mae:.2f}")

# 3. XGBoost
xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, 
                    max_depth=6, random_state=42)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test).clip(0, 125)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_mae = mean_absolute_error(y_test, xgb_preds)

print(f"✅ XGBoost           → RMSE: {xgb_rmse:.2f} | MAE: {xgb_mae:.2f}")

print("\n🏆 Model Comparison:")
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8}")
print("-" * 38)
print(f"{'Linear Regression':<20} {lr_rmse:>8.2f} {lr_mae:>8.2f}")
print(f"{'Random Forest':<20} {rf_rmse:>8.2f} {rf_mae:>8.2f}")
print(f"{'XGBoost':<20} {xgb_rmse:>8.2f} {xgb_mae:>8.2f}")

✅ Data ready for training!
Training samples: 20631
Test engines: 100

⏳ Training models... please wait...

✅ Linear Regression → RMSE: 22.43 | MAE: 17.64
✅ Random Forest     → RMSE: 17.93 | MAE: 13.22
✅ XGBoost           → RMSE: 18.72 | MAE: 14.06

🏆 Model Comparison:
Model                    RMSE      MAE
--------------------------------------
Linear Regression       22.43    17.64
Random Forest           17.93    13.22
XGBoost                 18.72    14.06


In [6]:
import joblib
import os

# Save all 3 models
joblib.dump(lr, '../models/linear_regression.pkl')
joblib.dump(rf, '../models/random_forest.pkl')
joblib.dump(xgb, '../models/xgboost.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Save predictions for dashboard
import json
predictions = {
    'engine_ids': list(range(1, 101)),
    'actual_rul': y_test.tolist(),
    'lr_preds': lr_preds.tolist(),
    'rf_preds': rf_preds.tolist(),
    'xgb_preds': xgb_preds.tolist()
}

with open('../data/processed/predictions.json', 'w') as f:
    json.dump(predictions, f)

print("✅ All models saved!")
print("✅ Predictions saved!")
print("\n📁 Saved in models/ folder:")
print("   - linear_regression.pkl")
print("   - random_forest.pkl")
print("   - xgboost.pkl")
print("   - scaler.pkl")

✅ All models saved!
✅ Predictions saved!

📁 Saved in models/ folder:
   - linear_regression.pkl
   - random_forest.pkl
   - xgboost.pkl
   - scaler.pkl
